# Real data can be complicated
We will look at a few difficulties that often occur when getting
our real world data into pandas. These involve dealing with data that are badly formatted and/or spread across multiple files.


In [1]:
import pandas as pd
awkward_file = "https://raw.githubusercontent.com/csbfx/advpy122-data/master/awkward_input.csv"

### Inconsistent data
Sometimes files are just a clean like a comma- or tab-delimited file. Such as having header lines, comment lines, or footer.

In [2]:
awkward = pd.read_csv(awkward_file)

ParserError: Error tokenizing data. C error: Expected 1 fields in line 3, saw 2


### Read lines in URL
The library **urllib** can be use to open up a URL and read lines by lines.

In [3]:
import urllib.request

### Read the first few lines
with urllib.request.urlopen(awkward_file) as response:
    for i in range(15):
        line = response.readline().decode('utf-8').strip()
        print(f"{i+1}: {line}")

1: Data file prepared on 4/10/2019
2: Original data taken from https://www.ncbi.nlm.nih.gov/
3: From left to right, the columns are:
4: species name
5: genome size in megabases
6: gc percentage
7: number of genes
8: publication year
9: assembly status
10: whether the genome belongs to an animal
11: whether the genome belongs to a plant
12: 
13: 
14: Arabidopsis thaliana,"119,669",36.0529,38_311,2001.0,Chromosome,False,True,,
15: Glycine max,"979,046",35.1153,59_847,2010.0,Chromosome,False,True,,


Reading in a csv file, you can pass the parameter `skiprows=<int>` to skip rows of the csv file before passing it into a DataFrame. Additionally, the parameter `header` can be use to define the header (=index, None).

In [4]:
awkward = pd.read_csv(awkward_file,
                      skiprows=<int>, # skip the first 11 rows
                      header=<int, None>, # no header line
                     )
awkward

,0,1,2,3,4,5,6,7,8,9
0,Arabidopsis thaliana,"119,669",36.0529,38_311,2001.0,Chromosome,False,True,NaN,NaN
1,Glycine max,"979,046",35.1153,59_847,2010.0,Chromosome,False,True,NaN,NaN
2,Medicago truncatula,"412,924",34.047,37_603,2011.0,Chromosome,False,True,NaN,NaN
3,Solanum lycopersicum,"828,349",35.6991,31_200,2010.0,Chromosome,no,yes,NaN,NaN
4,Hordeum vulgare,"4006,12",44.3,NaN,2019.0,Scaffold,False,True,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
3049,Homo sapiens,"2,097",45.8,NaN,2017.0,Scaffold,1,0,NaN,NaN
3050,Homo sapiens,"4,781",44.6,NaN,2017.0,Scaffold,1,0,NaN,NaN
3051,Homo sapiens,"4,799",44.6,missing,2017.0,Scaffold,True,False,NaN,NaN
3052,Homo sapiens,"2,79",34.8,NaN,2017.0,Scaffold,True,False,NaN,NaN


### Define header
Pandas by default assumes that the first line of data is the header and are the column names. If the first line of your data is not the header line that contains the column names, we can fix this by explicitly giving a list of column names that we want in the `names=[<colnames>]` argument.

In [5]:
my_columns = ["Species","Genome size","GC%","Genes",
              "Year","Status","Is animal","Is plant"]

awkward = pd.read_csv(awkward_file,
                      na_values=['-'], # set - as NaN
                      skiprows=11,
                      names=<[list of column name]>
                     )
awkward

Species Genome size     GC%  \
Arabidopsis thaliana               119,669  36.0529      38_311  2001.0   
Glycine max                        979,046  35.1153      59_847  2010.0   
Medicago truncatula                412,924   34.047      37_603  2011.0   
Solanum lycopersicum               828,349  35.6991      31_200  2010.0   
Hordeum vulgare                    4006,12     44.3         NaN  2019.0   
...                                             ...         ...     ...   
Homo sapiens                       2,097       45.8         NaN  2017.0   
                                   4,781       44.6         NaN  2017.0   
                                   4,799       44.6     missing  2017.0   
                                   2,79        34.8         NaN  2017.0   
This is version 2.5432 of the file NaN          NaN         NaN     NaN   

                                                 Genes   Year Status  \
Arabidopsis thaliana               119,669  Chromosome  False   True   
Glycine max                        979,046  Chromosome  False   True   
Medicago truncatula                412,924  Chromosome  False   True   
Solanum lycopersicum               828,349  Chromosome     no    yes   
Hordeum vulgare                    4006,12    Scaffold  False   True   
...                                                ...    ...    ...   
Homo sapiens                       2,097      Scaffold      1      0   
                                   4,781      Scaffold      1      0   
                                   4,799      Scaffold   True  False   
                                   2,79       Scaffold   True  False   
This is version 2.5432 of the file NaN             NaN    NaN    NaN   

                                            Is animal  Is plant  
Arabidopsis thaliana               119,669        NaN       NaN  
Glycine max                        979,046        NaN       NaN  
Medicago truncatula                412,924        NaN       NaN  
Solanum lycopersicum               828,349        NaN       NaN  
Hordeum vulgare                    4006,12        NaN       NaN  
...                                               ...       ...  
Homo sapiens                       2,097          NaN       NaN  
                                   4,781          NaN       NaN  
                                   4,799          NaN       NaN  
                                   2,79           NaN       NaN  
This is version 2.5432 of the file NaN            NaN       NaN  

[3054 rows x 8 columns]

# Clean data
### Empty columns
If you look at the raw data in the **awkward_input.csv**, you will see that all the data line ends with two commas. This is a fairly common occurence, especially in files that have been exported from spreadsheet programs such as Excel. These two extra commas cause pandas to assume that each row has 10 fields rather than 8, so when we give a list of only 8 column names, it uses the first two as the index. In our case, this means that all of the column names are offset by two places.

To fix this, we will need to explictly tell pandas with columns we want to use with the `usecols=[<colnames>]` argument.

In [6]:
awkward = pd.read_csv(awkward_file,
                      na_values=['-'],
                      skiprows=11,
                      names=my_columns,
                      usecols=<[list of column name]> # explicitly specify which columns to use
                     )
awkward

,Species,Genome size,GC%,Genes,Year,Status,Is animal,Is plant
0,Arabidopsis thaliana,"119,669",36.0529,38_311,2001.0,Chromosome,False,True
1,Glycine max,"979,046",35.1153,59_847,2010.0,Chromosome,False,True
2,Medicago truncatula,"412,924",34.047,37_603,2011.0,Chromosome,False,True
3,Solanum lycopersicum,"828,349",35.6991,31_200,2010.0,Chromosome,no,yes
4,Hordeum vulgare,"4006,12",44.3,NaN,2019.0,Scaffold,False,True
...,...,...,...,...,...,...,...,...
3049,Homo sapiens,"2,097",45.8,NaN,2017.0,Scaffold,1,0
3050,Homo sapiens,"4,781",44.6,NaN,2017.0,Scaffold,1,0
3051,Homo sapiens,"4,799",44.6,missing,2017.0,Scaffold,True,False
3052,Homo sapiens,"2,79",34.8,NaN,2017.0,Scaffold,True,False


### What if there is a footer following the data
Run `df.tail()`. Notice the last row in the DataFrame that it does not correspond to the data but rather is a footer comment. Many real life data files include a footer like this. We can tell pandas to skip it by passing `skipfooter=<int>` (on some versions, this may cause a `ParserWarning` , but it will still work. You can include the command `engine='python'` to avoid warning.

In [7]:
awkward.tail()

,Species,Genome size,GC%,Genes,Year,Status,Is animal,Is plant
3049,Homo sapiens,"2,097",45.8,NaN,2017.0,Scaffold,1,0
3050,Homo sapiens,"4,781",44.6,NaN,2017.0,Scaffold,1,0
3051,Homo sapiens,"4,799",44.6,missing,2017.0,Scaffold,True,False
3052,Homo sapiens,"2,79",34.8,NaN,2017.0,Scaffold,True,False
3053,This is version 2.5432 of the file,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
awkward = pd.read_csv(awkward_file,
                      na_values=['-'],
                      skiprows=11,
                      names=my_columns,
                      usecols=my_columns,
                      skipfooter=<int>,
                      engine='python', # avoid warning
                     )
awkward

,Species,Genome size,GC%,Genes,Year,Status,Is animal,Is plant
0,Arabidopsis thaliana,"119,669",36.0529,38_311,2001.0,Chromosome,False,True
1,Glycine max,"979,046",35.1153,59_847,2010.0,Chromosome,False,True
2,Medicago truncatula,"412,924",34.047,37_603,2011.0,Chromosome,False,True
3,Solanum lycopersicum,"828,349",35.6991,31_200,2010.0,Chromosome,no,yes
4,Hordeum vulgare,"4006,12",44.3,NaN,2019.0,Scaffold,False,True
...,...,...,...,...,...,...,...,...
3048,Homo sapiens,"4,898",44.6,missing,2017.0,Scaffold,yes,no
3049,Homo sapiens,"2,097",45.8,NaN,2017.0,Scaffold,1,0
3050,Homo sapiens,"4,781",44.6,NaN,2017.0,Scaffold,1,0
3051,Homo sapiens,"4,799",44.6,missing,2017.0,Scaffold,True,False


### Dealing with numbers contains other characters other than a dot as the decimal separator

In [10]:
# examine the data types with .info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3053 entries, 0 to 3052
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Species      3053 non-null   object 
 1   Genome size  3051 non-null   object 
 2   GC%          2897 non-null   object 
 3   Genes        1296 non-null   object 
 4   Year         3051 non-null   float64
 5   Status       3051 non-null   object 
 6   Is animal    3051 non-null   object 
 7   Is plant     3051 non-null   object 
dtypes: float64(1), object(7)
memory usage: 190.9+ KB


We notice that **Genome size**, **GC%**, and **Genes** should be numbers.

In [11]:
## Select which column to view in table
awkward[["Genome size", "GC%", "Genes"]]

,Genome size,GC%,Genes
0,"119,669",36.0529,38_311
1,"979,046",35.1153,59_847
2,"412,924",34.047,37_603
3,"828,349",35.6991,31_200
4,"4006,12",44.3,NaN
...,...,...,...
3048,"4,898",44.6,missing
3049,"2,097",45.8,NaN
3050,"4,781",44.6,NaN
3051,"4,799",44.6,missing


Looking at the values, we can see that these numbers have been written with a comma rather than a dot as the decimal separator (common practice in many countries). Setting the `decimal` argument will allow these numbers to be parsed correctly and turn our **Genome size** column into a floating point data type:

In [12]:
awkward = pd.read_csv(awkward_file,
                      na_values=['-'],
                      skiprows=11,
                      names=my_columns,
                      usecols=my_columns,
                      skipfooter=1,
                      decimal=<separator>, # specify decimal character
                      engine='python', # avoid warning
                     )
awkward.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3053 entries, 0 to 3052
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Species      3053 non-null   object 
 1   Genome size  3051 non-null   float64
 2   GC%          2897 non-null   object 
 3   Genes        1296 non-null   object 
 4   Year         3051 non-null   float64
 5   Status       3051 non-null   object 
 6   Is animal    3051 non-null   object 
 7   Is plant     3051 non-null   object 
dtypes: float64(2), object(6)
memory usage: 190.9+ KB


In [13]:
awkward

,Species,Genome size,GC%,Genes,Year,Status,Is animal,Is plant
0,Arabidopsis thaliana,119.669,36.0529,38_311,2001.0,Chromosome,False,True
1,Glycine max,979.046,35.1153,59_847,2010.0,Chromosome,False,True
2,Medicago truncatula,412.924,34.047,37_603,2011.0,Chromosome,False,True
3,Solanum lycopersicum,828.349,35.6991,31_200,2010.0,Chromosome,no,yes
4,Hordeum vulgare,4006.120,44.3,NaN,2019.0,Scaffold,False,True
...,...,...,...,...,...,...,...,...
3048,Homo sapiens,4.898,44.6,missing,2017.0,Scaffold,yes,no
3049,Homo sapiens,2.097,45.8,NaN,2017.0,Scaffold,1,0
3050,Homo sapiens,4.781,44.6,NaN,2017.0,Scaffold,1,0
3051,Homo sapiens,4.799,44.6,missing,2017.0,Scaffold,True,False


**GC%** should be numeric, however data type is object. Let's see if there is any non-numeric data in that column.

In [14]:
awkward["GC%"]

,GC%
0,36.0529
1,35.1153
2,34.047
3,35.6991
4,44.3
...,...
3048,44.6
3049,45.8
3050,44.6
3051,44.6


In [15]:
### Import regular expression library
import re

## Drop NA so we can explore the values in column
all_GC = awkward["GC%"].dropna().unique()
all_GC

## Are there values that are not numeric? Use python list comprehension
# Convert float 'i' to string before applying regex
non_num = [i for i in all_GC if not re.match("[.0-9]+", i)]
print(non_num)

['missing']


We can add 'missing' to our list of `na_values=[<missing values>]`

In [16]:
awkward = pd.read_csv(awkward_file,
                      na_values=[<missing values>],
                      skiprows=11,
                      names=my_columns,
                      usecols=my_columns,
                      skipfooter=1,
                      decimal=",",
                      engine='python',
                     )
awkward.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3053 entries, 0 to 3052
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Species      3053 non-null   object 
 1   Genome size  3051 non-null   float64
 2   GC%          2859 non-null   float64
 3   Genes        802 non-null    object 
 4   Year         3051 non-null   float64
 5   Status       3051 non-null   object 
 6   Is animal    3051 non-null   object 
 7   Is plant     3051 non-null   object 
dtypes: float64(3), object(5)
memory usage: 190.9+ KB


### Inconsistent thousands separators
**Genes** should be numeric. Let's explore the column and see why it is not consider a numeric.

In [17]:
awkward["Genes"].dtype
awkward["Genes"]

,Genes
0,38_311
1,59_847
2,37_603
3,31_200
4,NaN
...,...
3048,NaN
3049,NaN
3050,NaN
3051,NaN


The problem is the thousands separator, which is an underscore. Setting the `thousands=[<value>]` argument will allow pandas to parse these numbers properly.

In [18]:
awkward = pd.read_csv(awkward_file,
                      na_values=['-', 'missing'],
                      skiprows=11,
                      names=my_columns,
                      usecols=my_columns,
                      skipfooter=1,
                      engine='python',
                      decimal=",",
                      thousands=[<value>]
                     )
awkward.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3053 entries, 0 to 3052
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Species      3053 non-null   object 
 1   Genome size  3051 non-null   float64
 2   GC%          2859 non-null   float64
 3   Genes        802 non-null    float64
 4   Year         3051 non-null   float64
 5   Status       3051 non-null   object 
 6   Is animal    3051 non-null   object 
 7   Is plant     3051 non-null   object 
dtypes: float64(4), object(4)
memory usage: 190.9+ KB


Now that we have all our columns in the right dtypes, take a look at the numbers of values. We have 3053 rows, but five of the columns that we expect to have no missing data have 3051 values. This is suspicious! Let’s find the two rows with a missing size and take a look at the data:

In [19]:
awkward[awkward["Year"].isnull()]

,Species,Genome size,GC%,Genes,Year,Status,Is animal,Is plant
28,# the next two genomes belong to frogs,NaN,NaN,NaN,NaN,None,None,None
86,# three different Caenorhabditis species,NaN,NaN,NaN,NaN,None,None,None


### Comment lines in the data
These comment lines contain no commas, so the entire line has
ended up in the Species column. The remaining columns for those rows are either missing data (in the case of the numerical columns) or `None` (in the case of the string columns).

To fix this, we just have to tell pandas that lines begining with a hash symbol are comments by setting the `comment=[<value>]` argument:

In [ ]:
awkward = pd.read_csv(awkward_file,
                      na_values=['-', 'missing'],
                      skiprows=11,
                      names=my_columns,
                      usecols=my_columns,
                      skipfooter=1,
                      engine='python',
                      decimal=",",
                      thousands="_",
                      comment=[<values>] # lines starts with # is considered comment
                     )
awkward.info()

In [ ]:
awkward

### Inconsistent boolean values
Let’s take a look at the final two columns: **Is animal** and **Is plant**. These are examples of _boolean_ data type, i.e. True/False values. We can specify what data type we want for a given column by passing a `dtype` dict; `dtype={"Is animal": bool, "Is plant": bool}`

In [20]:
awkward = pd.read_csv(awkward_file,
                      na_values=['-', 'missing'],
                      skiprows=11,
                      names=my_columns,
                      usecols=my_columns,
                      skipfooter=1,
                      engine='python',
                      decimal=",",
                      thousands="_",
                      comment="#",
                      dtype={<key>:bool}
                     )
awkward.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3051 entries, 0 to 3050
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Species      3051 non-null   object 
 1   Genome size  3051 non-null   float64
 2   GC%          2859 non-null   float64
 3   Genes        802 non-null    float64
 4   Year         3051 non-null   float64
 5   Status       3051 non-null   object 
 6   Is animal    3051 non-null   object 
 7   Is plant     3051 non-null   object 
dtypes: float64(4), object(4)
memory usage: 190.8+ KB


In [ ]:
awkward.head()

In [ ]:
### Find the distinct observation and counts for True/False in the column Is animal
awkward["Is animal"].value_counts()

The reason for the preponderance of `True` values when we tried setting the data type as boolean is because of Python’s rule for determining boolean values. The rule for strings is simple: any non-empty string counts as `True`. So all of the values that are in the file are interpreted as `True`.

In [21]:
trueList = []
falseList = []
for val in awkward["Is animal"].unique():
    trueList.append(val)
    print(repr(val), bool(val)) #repr provides representation, bool is the boolean

print(trueList)
falseList =
trueList =

'False' True
'no' True
'"f"' True
'0' True
'1' True
'yes' True
'"t"' True
'True' True
['False', 'no', '"f"', '0', '1', 'yes', '"t"', 'True']
['False', 'no', '"f"', '0']
['1', 'yes', '"t"', 'True']


Let’s fix the boolean values by passing `true_values` and `false_values` to explicity tell pandas what values we want to interpret as true and false:

In [22]:
awkward = pd.read_csv(awkward_file,
                      na_values=['-', 'missing'],
                      skiprows=11,
                      names=my_columns,
                      usecols=my_columns,
                      skipfooter=1,
                      engine='python',
                      decimal=",",
                      thousands="_",
                      comment="#",
                      true_values = <trueList>,
                      false_values = <falseList>,
                      dtype={"Is animal": bool, "Is plant": bool}
                     )
awkward.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3051 entries, 0 to 3050
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Species      3051 non-null   object 
 1   Genome size  3051 non-null   float64
 2   GC%          2859 non-null   float64
 3   Genes        802 non-null    float64
 4   Year         3051 non-null   float64
 5   Status       3051 non-null   object 
 6   Is animal    3051 non-null   bool   
 7   Is plant     3051 non-null   bool   
dtypes: bool(2), float64(4), object(2)
memory usage: 149.1+ KB


In [ ]:
# check to see if we have a good mixture of True and False
awkward[["Is animal","Is plant"]].apply(lambda x: x.value_counts())

In [ ]:
awkward

# Combining multiple files
A very common obstacle to data analysis is when our data are spread out over multiple files. In the real world we often have
to bring data together from multiple sources before we can start our analysis.

### Concatenation
Let's look at the dataframe that contains the monthly rainfall data for London, Berlin, and Edinburgh.

In [23]:
import pandas as pd
london_data="https://raw.githubusercontent.com/csbfx/advpy122-data/master/London_daily_rain.csv"

berlin_data="https://raw.githubusercontent.com/csbfx/advpy122-data/master/Berlin_daily_rain.csv"

edinburgh_data="https://raw.githubusercontent.com/csbfx/advpy122-data/master/Edinburgh_daily_rain.csv"

london_rain = pd.read_csv(london_data)
london_rain.head()
#london_rain.info()

,Year,Month,Day of month,Day of year,Rainfall (mm)
0,1960,January,1,1,22.0
1,1960,January,2,2,23.0
2,1960,January,3,3,7.0
3,1960,January,4,4,0.0
4,1960,January,5,5,0.0


In [24]:
berlin_rain = pd.read_csv(berlin_data)
berlin_rain.head()
#berlin_rain.info()

,Year,Month,Day of month,Day of year,Rainfall (mm)
0,1960,January,1,1,4.0
1,1960,January,2,2,25.0
2,1960,January,3,3,0.0
3,1960,January,4,4,0.0
4,1960,January,5,5,104.0


In [25]:
edinburgh_rain = pd.read_csv(edinburgh_data)
edinburgh_rain.head()

,Year,Month,Day of month,Day of year,Rainfall (mm)
0,1960,January,1,1,26.0
1,1960,January,2,2,24.0
2,1960,January,3,3,7.0
3,1960,January,4,4,3.0
4,1960,January,5,5,4.0


Once we’ve got our three dataframes in memory, we can check that they have the same length and having the same columns:

In [26]:
# Check the length of each dataframe
all_dfs = [london_rain, berlin_rain, edinburgh_rain]


# Check the columns of each dataframe


Lengths: [21762, 21762, 21762]
Index(['Year', 'Month', 'Day of month', 'Day of year', 'Rainfall (mm)'], dtype='object')
Index(['Year', 'Month', 'Day of month', 'Day of year', 'Rainfall (mm)'], dtype='object')
Index(['Year', 'Month', 'Day of month', 'Day of year', 'Rainfall (mm)'], dtype='object')


Since all the columns are the same, we can combine these dataframes by simply putting them underneath each other to get one combined dataframe. The function that does the **concatenation** is `pd.concat`. We can pass a list of dataframes to concatenate

In [47]:
### Concatenate the dataframes via pd.concat([dfs])
all_rain =


,Year,Month,Day of month,Day of year,Rainfall (mm),City
0,1960,January,1,1,22.0,London
1,1960,January,2,2,23.0,London
2,1960,January,3,3,7.0,London
3,1960,January,4,4,0.0,London
4,1960,January,5,5,0.0,London
...,...,...,...,...,...,...
21757,2019,July,27,208,18.0,Edinburgh
21758,2019,July,28,209,0.0,Edinburgh
21759,2019,July,29,210,4.0,Edinburgh
21760,2019,July,30,211,41.0,Edinburgh


Although the data look fine from a glance at the big dataframe, we have run into a common problem where we have lost track of which rows belong to which city. In other words, if we take our big dataframe and select a single
day, we can not distinguish which row belonged to which dataframe.

In [28]:
all_rain[(all_rain["Year"] == 2015)
        & (all_rain["Month"] == "October")
        & (all_rain["Day of month"]== 29)
        ]

,Year,Month,Day of month,Day of year,Rainfall (mm)
20390,2015,October,29,302,70.0
20390,2015,October,29,302,0.0
20390,2015,October,29,302,71.0


We can see that we have three rainfall measurements, but no idea which city they belong to. To fix this, we need to add a city column to each individual dataframe before we concatenate them.

In [29]:
## Add a City column
london_rain["City"] = "London"
berlin_rain["City"] = "Berlin"
edinburgh_rain["City"] = "Edinburgh"

## Concatenate the DataFrames
all_rain = pd.concat(all_dfs)
all_rain

,Year,Month,Day of month,Day of year,Rainfall (mm),City
0,1960,January,1,1,22.0,London
1,1960,January,2,2,23.0,London
2,1960,January,3,3,7.0,London
3,1960,January,4,4,0.0,London
4,1960,January,5,5,0.0,London
...,...,...,...,...,...,...
21757,2019,July,27,208,18.0,Edinburgh
21758,2019,July,28,209,0.0,Edinburgh
21759,2019,July,29,210,4.0,Edinburgh
21760,2019,July,30,211,41.0,Edinburgh


In [30]:
### Filter DataFrame for October 29, 2015
all_rain[(all_rain["Year"] == 2015)
        & (all_rain["Month"] == "October")
        & (all_rain["Day of month"]== 29)
        ]

,Year,Month,Day of month,Day of year,Rainfall (mm),City
20390,2015,October,29,302,70.0,London
20390,2015,October,29,302,0.0,Berlin
20390,2015,October,29,302,71.0,Edinburgh


## Merging
If we have a situation that’s more complicated than either of the two outlined above, then we need to **merge**.

Unlike concatenating or adding single columns, merging allows us to combine two entire dataframes while having much more control over the process. For example, we can specify exactly how to find matching rows, what we want to do when there are missing rows, and what we want to do when there are multiple matching rows.

Let's use the eukaryote dataset and a file with the common names for the organisms:

In [40]:
import pandas as pd

common_name_data = "https://raw.githubusercontent.com/csbfx/advpy122-data/master/common_names.csv"
euk_data = "https://raw.githubusercontent.com/csbfx/advpy122-data/master/euk.tsv"

euk = pd.read_csv(euk_data, sep="\t")
names = pd.read_csv(common_name_data)

euk

,Species,Kingdom,Class,Size (Mb),GC%,Number of genes,Number of proteins,Publication year,Assembly status
0,Emiliania huxleyi CCMP1516,Protists,Other Protists,167.676000,64.5,38549,38554,2013,Scaffold
1,Arabidopsis thaliana,Plants,Land Plants,119.669000,36.0529,38311,48265,2001,Chromosome
2,Glycine max,Plants,Land Plants,979.046000,35.1153,59847,71219,2010,Chromosome
3,Medicago truncatula,Plants,Land Plants,412.924000,34.047,37603,41939,2011,Chromosome
4,Solanum lycopersicum,Plants,Land Plants,828.349000,35.6991,31200,37660,2010,Chromosome
...,...,...,...,...,...,...,...,...,...
8297,Saccharomyces cerevisiae,Fungi,Ascomycetes,3.993920,38.2,-,-,2017,Scaffold
8298,Saccharomyces cerevisiae,Fungi,Ascomycetes,0.586761,38.5921,155,298,1992,Chromosome
8299,Saccharomyces cerevisiae,Fungi,Ascomycetes,12.020400,38.2971,-,-,2018,Chromosome
8300,Saccharomyces cerevisiae,Fungi,Ascomycetes,11.960900,38.2413,-,-,2018,Chromosome


In [41]:
names

,scientific name,common name
0,Simonsiella muelleri,Scheibenbakterien
1,Simonsiella muelleri,Scheibenbakterien Muller 1911
2,Escherichia coli,E. coli
3,Rickettsia akari,rickettsialpox
4,Anaplasma phagocytophilum,agent of human granulocytic ehrlichiosis
...,...,...
14436,Riboviria,RNA viruses and viroids
14437,Neoheterocotyle quadrispinata,Yotsu-toge-iban-chu
14438,Arthroleptis lameerei,Lameere's squeaker
14439,Erebia albergana,almond-eyed ringlet butterfly


Our goal is to add a new column to our eukaryote dataframe that contains the common name.

Here is the challenge, each **Species** in the `euk` dataframe can have zero, one, or multiple **common names**, and a
**common name** from the `names` dataframe can belong to zero, one or multiple **Species**. This is where most of the
complication lies in merging: deciding what to do when we have multiple matches and missing data.

The method that we need is `merge`. We will have to tell pandas explicitly which columns we want to match. To do this, we refer to the dataframe on which we called the method (euk) as the right dataframe, and the one that we passed as the argument (names) as the left dataframe. These are conventional names, simply based on the order in which we write the variables. So, for our dataset, the *Species* column in the left dataframe matches the *scientific name* column from the right dataframe.

In [ ]:
merged = <dataframe1>.merge(<dataframe2>, left_on=<colnam1>, right_on=<colname2>)
merged

Additionally, we can see that where *Species* has multiple common names, we have ended up with multiple copies of each original row. For example, Arabidopsis thaliana has two common names:

In [42]:
names[names["scientific name"]== "Arabidopsis thaliana"]

,scientific name,common name
249,Arabidopsis thaliana,mouse-ear cress
250,Arabidopsis thaliana,thale-cress


In the merged dataframe we see that every Arabidopsis thaliana genome that appears in the original dataframe will appear twice in the merged dataframe; once with each common name.

Understanding this behavior is very important, so let’s summarize:
- genomes that have no matching common name are missing from the merged dataframe
- genomes that have multiple common names are duplicated in the merged dataframe

The easiest way to avoid having duplicated entries is to make sure that our names dataframe only contains a single common name for each scientific name, which we can do by calling `drop_duplicates()` and telling pandas which columns we want to avoid duplicates in.

In [43]:
names.drop_duplicates(subset=["scientific name"])

,scientific name,common name
0,Simonsiella muelleri,Scheibenbakterien
2,Escherichia coli,E. coli
3,Rickettsia akari,rickettsialpox
4,Anaplasma phagocytophilum,agent of human granulocytic ehrlichiosis
5,Neorickettsia risticii,equine monocytic ehrlichiosis agent
...,...,...
14435,Riboviria,RNA viruses
14437,Neoheterocotyle quadrispinata,Yotsu-toge-iban-chu
14438,Arthroleptis lameerei,Lameere's squeaker
14439,Erebia albergana,almond-eyed ringlet butterfly


In [45]:
# Let's do the merge again with unique common names
merged = euk.merge(names.drop_duplicates(subset=["scientific name"]),
                   left_on="Species",
                   right_on="scientific name")
merged

,Species,Kingdom,Class,Size (Mb),GC%,Number of genes,Number of proteins,Publication year,Assembly status,scientific name,common name
0,Arabidopsis thaliana,Plants,Land Plants,119.669000,36.0529,38311,48265,2001,Chromosome,Arabidopsis thaliana,mouse-ear cress
1,Glycine max,Plants,Land Plants,979.046000,35.1153,59847,71219,2010,Chromosome,Glycine max,soybeans
2,Hordeum vulgare,Plants,Land Plants,4006.120000,44.3,-,-,2019,Scaffold,Hordeum vulgare,barley
3,Oryza sativa Japonica Group,Plants,Land Plants,374.423000,43.5769,35219,42580,2015,Chromosome,Oryza sativa Japonica Group,Japonica rice
4,Triticum aestivum,Plants,Land Plants,14547.300000,46.0544,-,-,2018,Chromosome,Triticum aestivum,Canadian hard winter wheat
...,...,...,...,...,...,...,...,...,...,...,...
1993,Saccharomyces cerevisiae,Fungi,Ascomycetes,3.993920,38.2,-,-,2017,Scaffold,Saccharomyces cerevisiae,S. cerevisiae
1994,Saccharomyces cerevisiae,Fungi,Ascomycetes,0.586761,38.5921,155,298,1992,Chromosome,Saccharomyces cerevisiae,S. cerevisiae
1995,Saccharomyces cerevisiae,Fungi,Ascomycetes,12.020400,38.2971,-,-,2018,Chromosome,Saccharomyces cerevisiae,S. cerevisiae
1996,Saccharomyces cerevisiae,Fungi,Ascomycetes,11.960900,38.2413,-,-,2018,Chromosome,Saccharomyces cerevisiae,S. cerevisiae


We get a single output row for each Species that has a common name. The species that don't have a common name are now missing in the merged dataframe.

To put back the missing species, that is the ones that do not have matching common names, we need to take a look at the `how=` argument to merge. The merges that we have been doing so far is the default option, `how='inner'` which intersects the data.


In Pandas, `merge(how='inner')` returns only rows with matching keys in both right and left datafranes. The other method of merge joins union data and is called using `merge(how='outer')` which returns all rows from both, filling missing matches with null value, NaN.

In [44]:
merged = euk.merge(names.drop_duplicates(subset=["scientific name"]),
                   left_on="Species",
                   right_on="scientific name",
                   how=<inner, outer, left, right>)
merged

,Species,Kingdom,Class,Size (Mb),GC%,Number of genes,Number of proteins,Publication year,Assembly status,scientific name,common name
0,Arabidopsis thaliana,Plants,Land Plants,119.669000,36.0529,38311,48265,2001,Chromosome,Arabidopsis thaliana,mouse-ear cress
1,Glycine max,Plants,Land Plants,979.046000,35.1153,59847,71219,2010,Chromosome,Glycine max,soybeans
2,Hordeum vulgare,Plants,Land Plants,4006.120000,44.3,-,-,2019,Scaffold,Hordeum vulgare,barley
3,Oryza sativa Japonica Group,Plants,Land Plants,374.423000,43.5769,35219,42580,2015,Chromosome,Oryza sativa Japonica Group,Japonica rice
4,Triticum aestivum,Plants,Land Plants,14547.300000,46.0544,-,-,2018,Chromosome,Triticum aestivum,Canadian hard winter wheat
...,...,...,...,...,...,...,...,...,...,...,...
1993,Saccharomyces cerevisiae,Fungi,Ascomycetes,3.993920,38.2,-,-,2017,Scaffold,Saccharomyces cerevisiae,S. cerevisiae
1994,Saccharomyces cerevisiae,Fungi,Ascomycetes,0.586761,38.5921,155,298,1992,Chromosome,Saccharomyces cerevisiae,S. cerevisiae
1995,Saccharomyces cerevisiae,Fungi,Ascomycetes,12.020400,38.2971,-,-,2018,Chromosome,Saccharomyces cerevisiae,S. cerevisiae
1996,Saccharomyces cerevisiae,Fungi,Ascomycetes,11.960900,38.2413,-,-,2018,Chromosome,Saccharomyces cerevisiae,S. cerevisiae


Another method to merge is to use `merge(how='left')` which will include all the rows from the left dataframe, even the ones that have no matching row in the right. In our case, this means keeping all species, even the ones that have no common name.

In [46]:
merged = euk.merge(names.drop_duplicates(subset=["scientific name"]),
                   left_on="Species",
                   right_on="scientific name",
                   how=<inner, outer, left, right>
merged

,Species,Kingdom,Class,Size (Mb),GC%,Number of genes,Number of proteins,Publication year,Assembly status,scientific name,common name
0,Emiliania huxleyi CCMP1516,Protists,Other Protists,167.676000,64.5,38549,38554,2013,Scaffold,NaN,NaN
1,Arabidopsis thaliana,Plants,Land Plants,119.669000,36.0529,38311,48265,2001,Chromosome,Arabidopsis thaliana,mouse-ear cress
2,Glycine max,Plants,Land Plants,979.046000,35.1153,59847,71219,2010,Chromosome,Glycine max,soybeans
3,Medicago truncatula,Plants,Land Plants,412.924000,34.047,37603,41939,2011,Chromosome,NaN,NaN
4,Solanum lycopersicum,Plants,Land Plants,828.349000,35.6991,31200,37660,2010,Chromosome,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
8297,Saccharomyces cerevisiae,Fungi,Ascomycetes,3.993920,38.2,-,-,2017,Scaffold,Saccharomyces cerevisiae,S. cerevisiae
8298,Saccharomyces cerevisiae,Fungi,Ascomycetes,0.586761,38.5921,155,298,1992,Chromosome,Saccharomyces cerevisiae,S. cerevisiae
8299,Saccharomyces cerevisiae,Fungi,Ascomycetes,12.020400,38.2971,-,-,2018,Chromosome,Saccharomyces cerevisiae,S. cerevisiae
8300,Saccharomyces cerevisiae,Fungi,Ascomycetes,11.960900,38.2413,-,-,2018,Chromosome,Saccharomyces cerevisiae,S. cerevisiae
